In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

In [29]:
#Load data
df = pd.read_csv('/Users/preciousajilore/Documents/GitHub/torchmtlr/notebooks/prepostop.csv')

In [30]:
#Drop the rows where failure is equal to 2
df = df[df['failure'] != 2]

In [31]:
#Sanityy check lol
df['failure'].unique()

array([0., 1.])

In [32]:
df.columns

Index(['stxlocation', 'distal', 'penile', 'stxetiology', 'stxlength',
       '#strictures', 'charlsons', 'cormorbidity', 'diabetes', 'copd',
       'smoker', 'bmi35+', 'prevprocedure', '#prevprocedures', 'cysto', 'open',
       'ordate', 'urine', 'stx_length_1', 'stx_length_2', 'stxetiology_0',
       'stxetiology_1', 'stxetiology_2', 'stxetiology_3', 'stxetiology_4',
       'stxetiology_5', 'stxetiology_6', 'stxlocation_0', 'stxlocation_1',
       'stxlocation_2', 'stxlocation_3', 'stxlocation_4', 'stxlocation_5',
       'stxlocation_6', 'stx_length_1.1', 'stx_length_2.1', 'cysto_0.0',
       'cysto_1.0', 'cysto_2.0', 'cysto_3.0', 'abx', 'erectilepre', 'uti',
       'los(days)', 'spc', 'tissue', 'transection', 'urethroplasty',
       'cathremoval', 'cathdays', 'tissue_0.0', 'tissue_1.0', 'tissue_2.0',
       'tissue_3.0', 'tissue_4.0', 'tissue_5.0', 'urethroplasty_1.0',
       'urethroplasty_2.0', 'urethroplasty_3.0', 'urethroplasty_4.0',
       'urethroplasty_5.0', 'urethroplasty_6.0

In [33]:
#Drop columns that we wont use for prediction

"""

Index(['Unnamed: 0', 'distal', 'penile', 'stxlength', '#strictures',
       'charlsons', 'cormorbidity', 'diabetes', 'copd', 'smoker', 'bmi35+',
       'bmiexact', 'prevprocedure', '#prevprocedures', 'cysto', 'open',
       'ordate', 'urine', 'failure', 'patent', 'satisfaction',
       'datetofailureorfollowup', 'Date of Surgery', 'fu', 'time_to_event',
       'open_clean', 'stxlength_1', 'stxlength_2', 'stxlocation_0',
       'stxlocation_1', 'stxlocation_2', 'stxlocation_3', 'stxlocation_4',
       'stxlocation_5', 'stxlocation_6', 'stxetiology_0', 'stxetiology_1',
       'stxetiology_18', 'stxetiology_2', 'stxetiology_3', 'stxetiology_4',
       'stxetiology_5', 'stxetiology_6'],
      dtype='object')

"""
drop = ['ordate',"stxlocation","stxetiology","tissue","cysto","cathremoval","cathdays"]

df = df.drop(drop, axis=1)

In [34]:
#Sanity check 
df.columns

Index(['distal', 'penile', 'stxlength', '#strictures', 'charlsons',
       'cormorbidity', 'diabetes', 'copd', 'smoker', 'bmi35+', 'prevprocedure',
       '#prevprocedures', 'open', 'urine', 'stx_length_1', 'stx_length_2',
       'stxetiology_0', 'stxetiology_1', 'stxetiology_2', 'stxetiology_3',
       'stxetiology_4', 'stxetiology_5', 'stxetiology_6', 'stxlocation_0',
       'stxlocation_1', 'stxlocation_2', 'stxlocation_3', 'stxlocation_4',
       'stxlocation_5', 'stxlocation_6', 'stx_length_1.1', 'stx_length_2.1',
       'cysto_0.0', 'cysto_1.0', 'cysto_2.0', 'cysto_3.0', 'abx',
       'erectilepre', 'uti', 'los(days)', 'spc', 'transection',
       'urethroplasty', 'tissue_0.0', 'tissue_1.0', 'tissue_2.0', 'tissue_3.0',
       'tissue_4.0', 'tissue_5.0', 'urethroplasty_1.0', 'urethroplasty_2.0',
       'urethroplasty_3.0', 'urethroplasty_4.0', 'urethroplasty_5.0',
       'urethroplasty_6.0', 'failure', 'time_to_event'],
      dtype='object')

In [35]:
# Seperate the features and label
X = df.drop('failure', axis=1)
Y =  df['failure']


In [36]:
#Just incase we have categorical variables, we need to convert them to numerical variables using one hot encoding
X = pd.get_dummies(X, drop_first=True)

In [37]:
#Split into train and test sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.5, random_state=42)  

In [38]:
#Standarize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [39]:
#Define the models we want to use
models = {
   'LogisticRegression': LogisticRegression(class_weight="balanced",max_iter=1000, random_state=42),
   'RandomForest': RandomForestClassifier(class_weight = "balanced",random_state=42),
   'SVM': SVC(kernel = 'rbf', probability= True, random_state=42),
   'MLP Neural Net': MLPClassifier(hidden_layer_sizes = (32,16), max_iter= 300, random_state=42),
}

In [40]:
#Train and evaluate each model

for name, model in models.items():
    if name in ['SVM', 'MLP Neural Net']:
        model.fit(X_train_scaled, Y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, Y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(Y_test, y_pred)
    precision = precision_score(Y_test, y_pred)
    recall = recall_score(Y_test, y_pred)
    f1 = f1_score(Y_test, y_pred)
    auc = roc_auc_score(Y_test, y_prob)

    #Print metrics
    print(f"{name} Metrics:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC Score: {auc:.4f}")
    print(classification_report(Y_test, y_pred, digits=3))


/Users/preciousajilore/Documents/GitHub/torchmtlr/survivalenv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression Metrics:
Accuracy: 0.7231
Precision: 0.1833
Recall: 0.6404
F1 Score: 0.2850
AUC Score: 0.7376
              precision    recall  f1-score   support

         0.0      0.956     0.731     0.828       944
         1.0      0.183     0.640     0.285        89

    accuracy                          0.723      1033
   macro avg      0.569     0.686     0.557      1033
weighted avg      0.889     0.723     0.782      1033

RandomForest Metrics:
Accuracy: 0.9100
Precision: 0.2500
Recall: 0.0225
F1 Score: 0.0412
AUC Score: 0.7306
              precision    recall  f1-score   support

         0.0      0.915     0.994     0.953       944
         1.0      0.250     0.022     0.041        89

    accuracy                          0.910      1033
   macro avg      0.583     0.508     0.497      1033
weighted avg      0.858     0.910     0.874      1033

SVM Metrics:
Accuracy: 0.9138
Precision: 0.5000
Recall: 0.0112
F1 Score: 0.0220
AUC Score: 0.6534
              precision    r